# Run the experiments on Kaggle

Settings: **Accelerator: GPU T4 x2**, **Internet: on**. For the Llama
configurations add the secret `HF_TOKEN` (Add-ons, Secrets) of an account
that accepted the licence of `meta-llama/Llama-3.1-8B-Instruct`.

Pick the experiment in the second code cell, then use *Save Version, Save & Run
All*. Results are written to `/kaggle/working/results.zip`.


In [ ]:
import subprocess
REPO = "https://github.com/valeriobellandi/fedexp-poisoning.git"
subprocess.run(["git", "clone", "--depth", "1", REPO, "/kaggle/working/repo"], check=True)
print(subprocess.run(["git", "-C", "/kaggle/working/repo", "log", "--oneline", "-1"],
                     capture_output=True, text=True).stdout)


In [ ]:
# Configurations of each experiment, as in the paper.
EXPERIMENTS = {
    "pilot": ["pilot_benign", "pilot_a1", "pilot_a1_site"],
    "per_code_server": ["percode_benign", "percode_a1_global", "percode_a1_site"],
    "general_sharing": ["general_benign", "general_a1", "general6_benign",
                        "general6_a1_f1", "general6_a1_f2"],
    "support_threshold": ["shuf6_benign", "shuf6_a1_f1", "shuf6_a1_f2", "shuf6_a1_urban",
                          "shuf6_q2_benign", "shuf6_q2_a1_f1", "shuf6_q2_a1_f2"],
    "disputes": ["trust6_a1_f1", "trust6_a1_f2", "trustfb6_a1_f1", "trustfb6_a1_f2"],
    "adaptive": ["shuf6_a1_f2_n3", "trustfb6_a1_f2_n3", "shuf6_a1_f2_flood",
                 "trustfb6_a1_f2_flood"],
    "coverage_0_2": ["cov2_benign", "cov2_a1_f2", "cov2_trust_a1_f2", "cov2_trustfb_a1_f2"],
    "second_model": ["llama_shuf6_benign", "llama_shuf6_a1_f2", "llama_shuf6_q2_benign",
                     "llama_shuf6_q2_a1_f2", "llama_trust6_a1_f2", "llama_trustfb6_a1_f2",
                     "llama_shuf6_a1_f2_flood", "llama_trustfb6_a1_f2_flood"],
    "a2_a3": ["a2_benign", "a2_f2", "a2_trustfb_f2", "a2_implfb_benign", "a2_implfb_f2",
              "a3site_benign", "a3site_f2", "a3gen_f2", "a3gen_trustfb_f2"],
    "rounds": ["mr_benign", "mr_a1", "mr_a1_hit", "mr_a1_hit_fb", "mr_a1_own",
               "mr_trustfb_benign", "mr_trustfb_a1", "mr_trustfb_sticky_a1"],
    "real_data": ["heart_benign", "heart_a1", "heart_plain_benign", "heart_plain_a1",
                  "heart_ev_benign", "heart_ev_a1"],
}
EXPERIMENT = "pilot"      # <- choose here
SEEDS = [0, 1, 2, 3, 4]


In [ ]:
%cd /kaggle/working/repo
!pip -q install pyyaml pytest
# The pipeline is checked with the mock model before the GPU is used.
!PYTHONPATH=src python -m pytest tests -q


In [ ]:
import os
if EXPERIMENT == "second_model":
    from huggingface_hub import login
    from kaggle_secrets import UserSecretsClient
    login(token=UserSecretsClient().get_secret("HF_TOKEN"))
# Format check: the model must write rules and answers in the expected form.
# If it fails, the notebook stops here.
import subprocess
first = f"configs/{EXPERIMENTS[EXPERIMENT][0]}.yaml"
check = "fedexp.heart" if EXPERIMENT == "real_data" else "fedexp.formatcheck"
r = subprocess.run(["python", "-m", check, first], env={**os.environ, "PYTHONPATH": "src"})
if r.returncode != 0:
    raise RuntimeError("format check failed")


In [ ]:
configs = " ".join(f"configs/{c}.yaml" for c in EXPERIMENTS[EXPERIMENT])
seeds = " ".join(map(str, SEEDS))
!PYTHONPATH=src python -m fedexp.run --config {configs} --seeds {seeds} --out /kaggle/working/results


In [ ]:
!cd /kaggle/working && zip -qr results.zip results && ls -la results.zip
